# P09 · Evaluación de rentabilidad y riesgo
**QuantLab | Fundamentos financieros | Práctica 09/12**

## 1. Objetivos
Construir un evaluador de estrategias: capital, retorno, drawdown, Sharpe, Sortino, Calmar, profit factor, esperanza y riesgo de ruina. Distinguir métricas diarias de métricas de operaciones cerradas. Comparar con un benchmark y cuantificar el impacto de costes.

Los datos predeterminados son sintéticos y reproducibles. No representan Bitcoin ni una estrategia validada. No se envían órdenes.

Las funciones están en **`src/quantlab/p09/performance.py`** y sus pruebas en **`tests/test_performance.py`** (`pytest` desde la raíz del proyecto). Los ejercicios están resueltos en la sección 10.

## 2. Configuración
Utilizamos retornos diarios en decimal, días naturales (365 al año, mercado cripto), capital inicial sin flujos externos y un tipo libre de riesgo hipotético. En CSV se requiere `timestamp,net_return`, fechas UTC consecutivas, opcionalmente `gross_return,benchmark_return`. Un segundo CSV opcional de operaciones exige `net_pnl` en euros. Los días realmente en efectivo pueden tener retorno cero; los datos ausentes no deben imputarse automáticamente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import inspect
import json
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p09.performance import (break_even_daily_cost, cost_sensitivity, drawdown, equity_curve,
                                      load_daily_csv, manual_equity_drawdown, performance_metrics, recovery_return,
                                      risk_free_comparison, ruin_grid, ruin_table, simulate_ruin, split_stability,
                                      synthetic_daily, trade_log_to_pnl, trade_metrics, validate_daily)

DATA_MODE = 'synthetic'  # synthetic o csv
RETURNS_CSV = 'daily_returns.csv'
TRADES_CSV = None  # opcional: 'closed_trades.csv'
INITIAL_CAPITAL = 1000.
PERIODS_PER_YEAR = 365
RISK_FREE_ANNUAL = .02  # supuesto, no cotización
COST_BPS = 20.  # coste diario hipotético
SEED = 42
EXPORT = False
plt.rcParams.update({'figure.figsize':(9,3.4),'axes.grid':True,'grid.alpha':.2,'axes.spines.top':False,'axes.spines.right':False})

## 3. Datos
Simulamos 365 días con una operación cerrada diaria, reinversión total, costes proporcionales al capital inicial de cada jornada y un episodio adverso deliberado. El benchmark es otra serie sintética sin costes. En datos reales necesitamos valoraciones de posiciones abiertas a cierre diario, no solo beneficios realizados.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿la serie diaria es coherente?</b>

`validate_daily` se detiene con un error si:

- Hay menos de 2 días, fechas duplicadas o desordenadas.
- Falta algún día (la frecuencia debe ser exactamente diaria): **los huecos no se imputan**.
- Falta `net_return`, o algún retorno no es finito o es ≤ −100 %.

</div>

In [ ]:
print(inspect.getsource(validate_daily))

if DATA_MODE=='synthetic':
    daily, trades = synthetic_daily(SEED, COST_BPS, INITIAL_CAPITAL)
    provenance={'mode':DATA_MODE,'seed':SEED,'cost_bps':COST_BPS}
elif DATA_MODE=='csv':
    daily = load_daily_csv(RETURNS_CSV)
    trades=pd.read_csv(TRADES_CSV) if TRADES_CSV else None
    provenance={'mode':DATA_MODE,'returns_file':RETURNS_CSV,'trades_file':TRADES_CSV}
else: raise ValueError('Modo desconocido')

daily=validate_daily(daily)
print(provenance, '| días:',len(daily));display(daily.head().round(5))

## 4. Equity y Drawdown
$E_t=E_{t-1}(1+r_t)$; retorno acumulado $E_t/E_0-1$; drawdown $D_t=E_t/\max_{s\le t}E_s-1$. El máximo drawdown se expresa como magnitud positiva. Incluye el capital inicial al calcular el máximo histórico: si pierdes el primer día, debe contabilizarse. Para recuperar una caída de 50 % necesitas ganar 100 %.

In [ ]:
print(inspect.getsource(equity_curve))
print(inspect.getsource(drawdown))

equity=equity_curve(daily.net_return,INITIAL_CAPITAL);dd=drawdown(equity)
print(f'Capital final: {equity.iloc[-1]:.2f} EUR | MDD: {-dd.min():.2%}')
print(f'Máximo histórico: {equity[:dd.idxmin()].idxmax():%Y-%m-%d} | fondo del drawdown: {dd.idxmin():%Y-%m-%d}')
ax=equity.plot(label='Estrategia neta',color='#2463a0')
if 'benchmark_return' in daily: equity_curve(daily.benchmark_return,INITIAL_CAPITAL).plot(ax=ax,label='Benchmark',ls='--',color='#777777')
ax.set(title=f'Capital acumulado · {DATA_MODE}',xlabel='Fecha UTC',ylabel='EUR');ax.legend();plt.tight_layout();plt.show()

In [ ]:
fig,ax=plt.subplots();ax.fill_between(dd.index,dd.to_numpy()*100,0,color='#b65252',alpha=.7)
ax.set(title='Drawdown desde el máximo histórico',xlabel='Fecha UTC',ylabel='Drawdown (%)');plt.tight_layout();plt.show()

## 5. Métricas ajustadas por riesgo
Con $A$ periodos anuales y $n$ observaciones:

- CAGR: $(E_n/E_0)^{A/n}-1$.
- Volatilidad anualizada: $s(r)\sqrt A$ (desviación muestral).
- Sharpe: $\sqrt A\,\overline{r-r_f}/s(r)$, donde $r_f=(1+RF)^{1/A}-1$.
- Sortino: $\sqrt A\,\overline{r-r_f}/\sqrt{\operatorname{mean}(\min(r-r_f,0)^2)}$. Incluye todos los periodos en el denominador.
- Calmar: CAGR / máximo drawdown observado.

Las anualizaciones son convenciones; la dependencia temporal puede distorsionar su interpretación. Un denominador nulo produce NaN. Un CAGR negativo no se transforma en positivo al dividir por el MDD.

In [ ]:
print(inspect.getsource(performance_metrics))

metrics=pd.DataFrame({'strategy_net':performance_metrics(daily.net_return,INITIAL_CAPITAL,PERIODS_PER_YEAR,RISK_FREE_ANNUAL)})
if 'gross_return' in daily: metrics['strategy_gross']=performance_metrics(daily.gross_return,INITIAL_CAPITAL,PERIODS_PER_YEAR,RISK_FREE_ANNUAL)
if 'benchmark_return' in daily: metrics['benchmark']=performance_metrics(daily.benchmark_return,INITIAL_CAPITAL,PERIODS_PER_YEAR,RISK_FREE_ANNUAL)
display(metrics.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La estrategia pierde más de la mitad del capital; el benchmark gana</b>

| Métrica | Estrategia neta | Estrategia bruta | Benchmark |
|---|---|---|---|
| Capital final | **444,47 €** | 923,17 € | 1.134,36 € |
| Retorno acumulado / CAGR | **−55,6 %** | −7,7 % | +13,4 % |
| Volatilidad anualizada | 19,4 % | 19,4 % | 32,4 % |
| Máximo drawdown | **57,1 %** | 18,9 % | 24,7 % |
| Sharpe | −4,17 | −0,42 | 0,49 |
| Sortino | −4,99 | −0,58 | 0,72 |
| Calmar | −0,97 | −0,41 | 0,54 |

- **Los costes destruyen la estrategia:** 20 bps diarios suman un 73 % al año (20 × 365 = 7.300 bps) y, compuestos sobre 365 días, restan casi 48 puntos de retorno (−7,7 % bruto frente a −55,6 % neto).
- **Ni en bruto gana:** el episodio adverso de 10 días (−1,3 % diario) cuesta en torno a un 13 %, y anula el alfa bruto de la simulación.
- **El drawdown nunca se recupera:** el máximo histórico fue el 17 de enero y el fondo, el último día del año.
- La estrategia tiene **menos volatilidad** que el benchmark (19 % frente a 32 %, por su beta de 0,35), pero eso no la hace mejor: el Sharpe ajusta por riesgo, y sigue siendo muy negativo.

</div>

## 6. Métricas por operación cerrada
Una estrategia que gana nueve veces 1 € y pierde una vez 20 € tiene 90 % de acierto, pero pierde 11 €. Esperanza = media del PnL neto (−1,10 € en ese caso). Profit Factor = suma ganancias / valor absoluto de suma pérdidas. Payoff = ganancia media / pérdida media absoluta. Las operaciones a cero permanecen en los denominadores de probabilidades. Si no hay pérdidas, un profit factor infinito no demuestra una estrategia perfecta.

In [ ]:
print(inspect.getsource(trade_metrics))

if trades is not None:
    trades['net_pnl']=pd.to_numeric(trades.net_pnl,errors='raise');tm=trade_metrics(trades.net_pnl);display(tm.round(4))
    ax=trades.net_pnl.hist(bins=30,color='#2463a0');ax.axvline(tm.expectancy,color='#a73535',ls='--',label='Media')
    ax.set(title='Distribución de PnL de operaciones cerradas',xlabel='PnL neto (EUR)',ylabel='Número de operaciones');ax.legend();plt.tight_layout();plt.show()
else: print('Sin registro de operaciones: métricas por trade no disponibles.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Por operación: gana menos veces y gana menos cuando gana</b>

365 operaciones: **42,5 %** ganadoras, ganancia media 5,03 € frente a pérdida media 6,36 € (payoff **0,79**), **profit factor 0,58** y esperanza **−1,52 €** por operación. La suma (−555,53 €) coincide exactamente con la caída del capital (1.000 → 444,47 €): las métricas por operación y las diarias están reconciliadas.

</div>

## 7. Costes y fragilidad
La sensibilidad resta un coste hipotético al retorno bruto diario (no al neto). Este modelo equivale a aproximar los costes como proporción del capital por jornada. Para datos reales utiliza los costes de P08 y no vuelvas a restar spread o comisión si ya están incluidos.

In [ ]:
print(inspect.getsource(cost_sensitivity))

if 'gross_return' in daily:
    sensitivity=cost_sensitivity(daily.gross_return,[0,5,10,20,30,40])
    display(sensitivity.round(3));ax=sensitivity.plot(x='cost_bps_per_day',y='total_return_pct',marker='o',legend=False)
    ax.axhline(0,color='gray',ls='--');ax.set(title='Sensibilidad a costes diarios hipotéticos',xlabel='Coste (bps/día)',ylabel='Retorno acumulado (%)');plt.tight_layout();plt.show()
else: print('Sin retornos brutos: no se recalculan costes.')

## 8. Riesgo de ruina: Monte Carlo por bootstrap de bloques
Definimos ruina como alcanzar el 50 % del capital inicial en algún momento de los siguientes 180 días. Extraemos bloques contiguos de 5 retornos con reemplazo para preservar parte de la dependencia temporal y simulamos 3.000 trayectorias por exposición. Una exposición de 150 % es un escenario idealizado sin financiación ni reglas de margen; no una ejecución real. El capital cero es absorbente. Las probabilidades son condicionales al modelo y a la muestra; no cubren todos los eventos extremos posibles.

In [ ]:
print(inspect.getsource(simulate_ruin))

risk_table = ruin_table(daily.net_return, [.25,.5,1.,1.5], INITIAL_CAPITAL, seed=SEED, block=min(5,len(daily)))
display(risk_table.round(4))
_,paths=simulate_ruin(daily.net_return,exposure=1,seed=SEED,block=min(5,len(daily)))
q=np.quantile(paths,[.05,.5,.95],axis=0)*INITIAL_CAPITAL
fig,ax=plt.subplots();days=np.arange(paths.shape[1]);ax.fill_between(days,q[0],q[2],alpha=.22,color='#2463a0',label='Percentiles 5–95')
ax.plot(days,q[1],color='#2463a0',label='Mediana');ax.axhline(INITIAL_CAPITAL*.5,color='#a73535',ls='--',label='Umbral de ruina')
ax.set(title='Escenarios bootstrap · exposición 100 %',xlabel='Día simulado',ylabel='Capital (EUR)');ax.legend();plt.tight_layout();plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Costes y ruina</b>

**Sensibilidad a costes diarios:** 0 bps → −7,7 %; 5 → −23,1 %; 10 → −35,9 %; 20 → −55,6 %; 30 → −69,2 %; 40 → −78,6 %. Cada 10 bps diarios adicionales restan entre 9,5 y 28 puntos de retorno anual (más al principio, porque luego queda menos capital sobre el que perder).

**Riesgo de ruina** (tocar el 50 % del capital en 180 días, bloques de 5):

| Exposición | Riesgo de ruina | Capital final mediano |
|---|---|---|
| 25 % | 0,0 % | 906,53 € |
| 50 % | 0,0 % | 820,77 € |
| 100 % | **3,1 %** | 670,51 € |
| 150 % | **40,2 %** | 544,70 € |

- Con una esperanza negativa, **más exposición solo acelera la pérdida**: el capital mediano cae con cualquier exposición, y el riesgo de ruina se dispara con el apalancamiento.
- Con 25 % y 50 %, ninguna de las 3.000 trayectorias tocó el umbral en 180 días, pero el capital sigue cayendo: con más tiempo, la ruina llegaría (ver E6).

</div>

## 9. Validación matemática
Casos conocidos: pérdida inicial, recuperación, operaciones neutras, ausencia de pérdidas, Sharpe indefinido y exposición cero. Las pruebas verifican cálculos, no demuestran rentabilidad.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: casos conocidos y reconciliación entre métricas diarias y por operación
r=pd.Series([-.1,.2]);e=equity_curve(r,1000)
assert np.allclose(e,[1000,900,1080])
assert np.isclose(drawdown(e).min(),-.1)
assert np.isclose(performance_metrics(r).cumulative_return,.08)
t=trade_metrics([10,-5,0,15,-10])
assert np.isclose(t.win_rate,.4)
assert np.isclose(t.profit_factor,25/15)
assert np.isclose(t.expectancy,2)
assert np.isclose(t.payoff_ratio,12.5/7.5)
assert np.isinf(trade_metrics([1,2]).profit_factor)
assert np.isnan(trade_metrics([0,0]).profit_factor)
assert np.isnan(performance_metrics([0.,0.]).sharpe_ratio)
assert simulate_ruin([-.1,.1],exposure=0,block=1,paths=20)[0]==0
assert simulate_ruin([-.9,-.9],block=1,horizon=1,paths=20)[0]==1
if DATA_MODE=='synthetic':
    assert np.isclose(trades.net_pnl.sum(),equity.iloc[-1]-INITIAL_CAPITAL)
    assert np.allclose(trades.gross_pnl-trades.cost,trades.net_pnl)
print('12 comprobaciones generales superadas; 2 reconciliaciones adicionales en modo sintético.')

## 10. Ejercicios
1. **Equity y drawdown:** implementa ambas funciones sin `cumprod` ni `cummax`. Valida [−10 %, +20 %] desde 1.000 €.
2. **Recuperación:** calcula qué retorno necesitas para recuperar pérdidas de 10 %, 25 % y 50 %. Explica la asimetría.
3. **Win rate:** compara `[1]*9+[-20]` con `[10]*3+[-2]*7`. ¿Cuál tiene mayor esperanza y cuál mayor acierto?
4. **Sharpe y Sortino:** cambia el tipo libre de riesgo a 0 % y 5 %. ¿Qué sucede con retornos constantes?
5. **Costes:** encuentra el menor coste no negativo que hace perder dinero (si ya pierde sin costes, explícalo).
6. **Riesgo de ruina:** prueba umbral 70 %, horizonte 365 y bloques 1 y 10. ¿Por qué 0 de 3.000 simulaciones no implica riesgo nulo?
7. **Integración:** importa resultados reales de tu estrategia. Separa las operaciones cerradas de la valoración diaria de la cartera.
8. **Estabilidad:** divide cronológicamente la serie en dos y compara Sharpe y MDD. No cambies los parámetros mirando el tramo final.

### E1 · Equity y drawdown sin `cumprod` ni `cummax`

In [ ]:
print(inspect.getsource(manual_equity_drawdown))

e1 = manual_equity_drawdown([-.10, .20], 1000)
display(e1)
assert np.allclose(e1.equity, [1000, 900, 1080]) and np.allclose(e1.drawdown, [0, -.1, 0])
manual = manual_equity_drawdown(daily.net_return, INITIAL_CAPITAL)
assert np.allclose(manual.equity, equity) and np.allclose(manual.drawdown, dd)
print('E1 verificado: coincide con equity_curve y drawdown en el ejemplo y en los 365 días.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Capital y drawdown paso a paso</b>

| Día | Retorno | Capital | Máximo histórico | Drawdown |
|---|---|---|---|---|
| 0 | — | 1.000 € | 1.000 € | 0 % |
| 1 | −10 % | 900 € | 1.000 € | **−10 %** |
| 2 | +20 % | 1.080 € | 1.080 € | 0 % |

- El capital se actualiza multiplicando (1.000 × 0,9 = 900; 900 × 1,2 = 1.080), no sumando retornos: −10 % + 20 % daría +10 %, pero el resultado real es +8 %.
- El máximo histórico **empieza en el capital inicial**. Si empezara en el primer cierre (900), la pérdida del primer día no aparecería como drawdown.
- La implementación con bucle da exactamente lo mismo que `equity_curve` y `drawdown` en los 365 días.

</div>

### E2 · Retorno necesario para recuperar una pérdida

In [ ]:
print(inspect.getsource(recovery_return))

losses = np.array([.10, .25, .50, .75, .90])
display(pd.DataFrame({'pérdida': losses, 'retorno necesario': recovery_return(losses)}).style.format('{:.1%}'))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Las pérdidas pesan más que las ganancias</b>

| Pérdida | Retorno necesario para recuperar |
|---|---|
| 10 % | **+11,1 %** |
| 25 % | **+33,3 %** |
| 50 % | **+100 %** |
| 75 % | +300 % |
| 90 % | +900 % |

- Retorno necesario = 1 / (1 − pérdida) − 1. Tras perder L, queda (1 − L) del capital, y la ganancia se aplica sobre esa base **más pequeña**.
- La asimetría crece muy rápido: una pérdida del 10 % se recupera con algo más del 10 %, pero una del 50 % exige duplicar el capital. Por eso el drawdown importa tanto: la estrategia neta (MDD del 57 %) necesitaría ganar un **+133 %** para volver al máximo.

</div>

### E3 · Acierto frente a esperanza

In [ ]:
e3 = pd.DataFrame({'[1]×9 + [−20]': trade_metrics([1]*9 + [-20]), '[10]×3 + [−2]×7': trade_metrics([10]*3 + [-2]*7)})
display(e3.round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Acertar mucho no es ganar</b>

| | 9 × +1 € y 1 × −20 € | 3 × +10 € y 7 × −2 € |
|---|---|---|
| Acierto | **90 %** | 30 % |
| Payoff | 0,05 | **5,0** |
| Profit factor | 0,45 | **2,14** |
| Esperanza | −1,10 € | **+1,60 €** |
| PnL total | −11 € | **+16 €** |

- La primera tiene **más acierto**; la segunda, **más esperanza**. La que gana dinero es la segunda.
- La esperanza combina las dos cosas: acierto × ganancia media − fallo × pérdida media. Con un 90 % de acierto: 0,9 × 1 − 0,1 × 20 = −1,10 €. Con un 30 %: 0,3 × 10 − 0,7 × 2 = +1,60 €.
- Una estrategia que gana poco a menudo y pierde mucho de vez en cuando (vender seguros, por ejemplo) parece muy buena durante mucho tiempo, hasta que llega la pérdida grande.

</div>

### E4 · Sharpe y Sortino con distintos tipos libres de riesgo

In [ ]:
print(inspect.getsource(risk_free_comparison))

display(pd.concat({'neta': risk_free_comparison(daily.net_return, (0., .02, .05), PERIODS_PER_YEAR),
                   'benchmark': risk_free_comparison(daily.benchmark_return, (0., .02, .05), PERIODS_PER_YEAR)}).round(4))
print("Retorno constante de 10 bps diarios (≈ 44 % anual):")
display(risk_free_comparison(pd.Series([.001] * 30), (0., .02, .05)).round(4))
print("Retorno constante de 1 bp diario (≈ 3,7 % anual), por debajo de un tipo libre de riesgo del 5 %:")
display(risk_free_comparison(pd.Series([.0001] * 30), (0., .05)).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · El tipo libre de riesgo desplaza los ratios; con retornos constantes, el Sharpe no existe</b>

| Serie | Sharpe con rf 0 % | 2 % | 5 % |
|---|---|---|---|
| Estrategia neta | −4,07 | −4,17 | −4,32 |
| Benchmark | 0,55 | 0,49 | 0,40 |

- Subir el tipo libre de riesgo **resta** el mismo exceso diario a todos los días, así que el Sharpe y el Sortino bajan. El efecto es pequeño frente a la volatilidad (un 5 % anual son 1,3 bps diarios).
- **Con retornos constantes,** la desviación típica es 0 y el **Sharpe queda indefinido** (NaN), con cualquier tipo libre de riesgo: no hay riesgo que dividir. El código devuelve NaN en lugar de un número enorme que parecería excelente.
- El **Sortino** también queda indefinido si el retorno constante supera al tipo libre de riesgo (ningún día queda por debajo). Pero con un retorno constante de 1 bp diario frente a un 5 % anual, todos los días quedan por debajo, y el Sortino vale **−√365 ≈ −19,1**.

</div>

### E5 · Menor coste que hace perder dinero

In [ ]:
print(inspect.getsource(break_even_daily_cost))

e5 = pd.DataFrame({'estrategia (bruto)': break_even_daily_cost(daily.gross_return),
                   'benchmark (referencia)': break_even_daily_cost(daily.benchmark_return)}).T
display(e5)

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · La estrategia ya pierde sin costes</b>

- **Estrategia:** el retorno bruto acumulado es **−7,7 %** con coste cero. El menor coste no negativo que hace perder dinero es **0 bps**: ya pierde antes de pagar nada. Reducir costes no la arregla; el problema está en la señal (y en el episodio adverso).
- **Referencia (benchmark):** su retorno es +13,4 % sin costes, y deja de ganar con un coste de **3,45 bps diarios**. Es muy poco: menos que una sola comisión taker de 10 bps al día. Con el coste del modelo (20 bps diarios), incluso el benchmark perdería.

</div>

### E6 · Umbral, horizonte y tamaño de bloque

In [ ]:
print(inspect.getsource(ruin_grid))

e6 = ruin_grid(daily.net_return, (.5, .7), (180, 365), (1, 5, 10), exposure=1., paths=3000, seed=SEED)
display(e6.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · La ruina depende de cómo se defina</b>

Exposición 100 %, 3.000 trayectorias:

| Umbral | Horizonte | Bloque 1 | Bloque 5 | Bloque 10 |
|---|---|---|---|---|
| 50 % | 180 días | 1,9 % | 3,1 % | 3,4 % |
| 50 % | 365 días | 76,5 % | 74,0 % | 74,7 % |
| **70 %** | 180 días | 69,1 % | 66,8 % | 66,0 % |
| **70 %** | **365 días** | **99,4 %** | 98,6 % | 98,6 % |

- **Umbral y horizonte mandan:** con una esperanza negativa, el capital se desliza hacia abajo. Exigir no caer del 70 % o mirar un año en lugar de seis meses convierte un 2–3 % de ruina en un **99 %**.
- **El bloque cambia poco aquí:** los retornos sintéticos casi no tienen dependencia temporal, salvo el episodio adverso. Los bloques más largos conservan rachas como esa y suben un poco la ruina a 180 días (1,9 % → 3,4 %).
- **¿Por qué 0 de 3.000 no es riesgo nulo?** Porque el bootstrap solo puede recombinar los retornos que **ya ocurrieron**: no genera caídas mayores que la peor de la muestra, ni rachas más largas, ni eventos nuevos (un hackeo, un colapso del exchange). Además, con 3.000 trayectorias, una probabilidad real del 0,05 % daría de media 1,5 ruinas, y aun así saldrían 0 en un 22 % de las simulaciones. Lo único que dice "0 de 3.000" es que la probabilidad **con este modelo y esta muestra** es, con un 95 % de confianza, menor que un 0,1 % (regla del tres: 3 / 3.000).

</div>

### E7 · Integración con resultados reales (P05 sobre Kraken)
Usamos la estrategia de P05 (N = 20, h = 5, 20 bps por operación) sobre las últimas 720 velas de 1 minuto de Kraken y convertimos su registro de operaciones de validación en PnL en euros. Cambia `LIVE_E7` a `True` para repetirlo. Si la conexión falla, la celda se detiene con el error.

In [ ]:
print(inspect.getsource(trade_log_to_pnl))

LIVE_E7 = False

if LIVE_E7:
    from quantlab.p05.signals import (add_target, build_features, fetch_kraken_ohlc, regularize,
                                      split_train_test, strategy_table)
    live_close, live_prov = fetch_kraken_ohlc('XBTEUR', 1, '1min')
    live_close = regularize(live_close, '1min')
    print(live_prov['received_at'], '|', live_close.index.min(), '—', live_close.index.max())
    live_train, live_test, _ = split_train_test(add_target(build_features(live_close, 20), 5), .65)
    _, live_logs = strategy_table({'desarrollo': live_train, 'validación': live_test}, 20., 1.5)
    rows = {}
    for rule in ['momentum', 'mean_reversion', 'periodic_long']:
        log = trade_log_to_pnl(live_logs[('validación', rule)], INITIAL_CAPITAL)
        assert np.isclose(INITIAL_CAPITAL + log.net_pnl.sum(), INITIAL_CAPITAL * log.equity.iloc[-1])
        rows[f'{rule} · neto'] = trade_metrics(log.net_pnl)
        rows[f'{rule} · bruto'] = trade_metrics(log.gross_pnl)
    display(pd.DataFrame(rows).T[['trade_count', 'win_rate', 'payoff_ratio', 'profit_factor', 'expectancy', 'net_pnl']].round(3))
else:
    print("LIVE_E7 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Operaciones cerradas reales: sí; Sharpe diario: no</b>

**Datos:** P05 sobre 720 velas de 1 minuto de Kraken BTC/EUR (19-11-2026, de 08:50 a 20:49 UTC), periodo de validación (16:39–20:44 UTC), 20 bps por operación y 1.000 € iniciales con reinversión total.

| Regla | Operaciones | Acierto neto | Profit factor bruto | Profit factor neto | Esperanza neta | PnL neto |
|---|---|---|---|---|---|---|
| Momentum | 15 | 26,7 % | 0,57 | 0,16 | −2,75 € | −41,30 € |
| Mean reversion | 18 | 11,1 % | **1,43** | **0,15** | −1,72 € | −31,03 € |
| Periodic long | 41 | 9,8 % | 0,61 | 0,09 | −2,29 € | −93,72 € |

- La suma del PnL de cada regla **coincide exactamente** con la variación de su capital (por ejemplo, momentum: 1.000 → 958,70 €).
- **Mean reversion parece rentable en bruto** (profit factor 1,43, +4,45 €), pero los 20 bps por operación lo convierten en 0,15. Es el mismo mensaje de P08: el coste decide.
- **Operaciones cerradas frente a valoración diaria:** con estos datos se pueden calcular las métricas **por operación**, pero **no** un Sharpe diario. Para eso haría falta el valor de la cartera (efectivo + posición abierta valorada al cierre) **cada día**, y aquí solo hay 4 horas de un único día. Inventar una serie diaria a partir de operaciones de 5 minutos, o anualizar el Sharpe de 15 operaciones como si fueran días, daría números sin sentido.

</div>

### E8 · Estabilidad en dos tramos

In [ ]:
print(inspect.getsource(split_stability))

e8 = pd.concat({'neta': split_stability(daily.net_return, INITIAL_CAPITAL, PERIODS_PER_YEAR, RISK_FREE_ANNUAL),
                'bruta': split_stability(daily.gross_return, INITIAL_CAPITAL, PERIODS_PER_YEAR, RISK_FREE_ANNUAL)}, axis=1)
display(e8.loc[['cumulative_return', 'annualized_volatility', 'max_drawdown', 'sharpe_ratio', 'sortino_ratio']].round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Mala en los dos tramos, y por eso la descartaríamos</b>

Mismos parámetros en ambos tramos (sin ajustar nada mirando el segundo):

| | Neta: 1.er tramo | Neta: 2.º tramo | Bruta: 1.er tramo | Bruta: 2.º tramo |
|---|---|---|---|---|
| Retorno | −35,4 % | −31,2 % | −7,0 % | −0,7 % |
| Máximo drawdown | 38,2 % | 33,8 % | 17,8 % | 10,3 % |
| Sharpe | −4,62 | −3,74 | −0,78 | −0,07 |

- La estrategia neta es **consistentemente mala**: pierde en las dos mitades, con un Sharpe muy negativo en ambas.
- El primer tramo es peor porque contiene el episodio adverso (días 120–129). En bruto, el segundo tramo es casi neutro (−0,7 %): sin el episodio y sin costes, la estrategia apenas se mueve.
- Que el segundo tramo sea "menos malo" **no** es motivo para ajustar parámetros: hacerlo mirando ese tramo lo convertiría en parte del desarrollo, y haría falta un tercer periodo nunca visto para evaluar.

</div>

### Integración P05 y P08
En P05, `trade_logs[(periodo, regla)]` contiene `entry_time`, `exit_time`, `gross_return`, `net_return`. Para reinversión total sin solapamiento, `trade_log_to_pnl(log, INITIAL_CAPITAL)` calcula `capital_antes = INITIAL_CAPITAL * (1 + log.net_return).cumprod().shift(1, fill_value=1)` y `log['net_pnl'] = capital_antes * log.net_return` (ver E7). Exporta el registro seleccionado, no el CSV de señales (incluye operaciones que no se ejecutan).

Para Sharpe diario necesitas registrar efectivo y valor de posiciones abiertas por día. No inventes ese registro a partir de operaciones de duración irregular. Puedes ejecutar `trade_metrics(log.net_pnl)` por separado. P08 aporta los costes; no vuelvas a descontarlos.

In [ ]:
if EXPORT:
    out=Path('quantlab_exports')/f"p09_{DATA_MODE}_{pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ')}"
    out.mkdir(parents=True,exist_ok=True);daily.to_csv(out/'daily_returns.csv');metrics.to_csv(out/'performance.csv')
    risk_table.to_csv(out/'ruin_scenarios.csv',index=False);equity.to_csv(out/'equity.csv');dd.to_csv(out/'drawdown.csv')
    if trades is not None: trades.to_csv(out/'closed_trades.csv',index=False);tm.to_csv(out/'trade_metrics.csv')
    (out/'metadata.json').write_text(json.dumps({**provenance,'capital':INITIAL_CAPITAL,'periods_per_year':PERIODS_PER_YEAR,'rf_annual':RISK_FREE_ANNUAL,'ruin_threshold':.5,'ruin_horizon':180,'block':min(5,len(daily)),'paths':3000,'seed':SEED},indent=2),encoding='utf-8')
    print(out)
else: print('Activa EXPORT para guardar datos, métricas y metadatos.')

## 11. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P09</b>

**¿Qué estrategia aceptarías o descartarías y por qué?**
**Descartaría la estrategia sintética.** Pierde un 55,6 % neto (CAGR −55,6 %, MDD 57,1 %, Sharpe −4,17), pierde en los dos tramos cronológicos y pierde incluso **sin costes** (−7,7 %). Su profit factor es 0,58 y su esperanza, −1,52 € por operación. Con exposición del 150 %, tiene un 40 % de probabilidad de perder la mitad del capital en seis meses. El benchmark se comporta mejor (+13,4 %, Sharpe 0,49), pero con datos sintéticos no lo aceptaría como inversión: solo muestra que esta estrategia es peor que no hacer nada especial.

**¿Qué costes o riesgos no contempla el modelo?**
- Costes reales de ejecución (spread, slippage, comisiones por nivel, de P08) en lugar de un coste fijo por día.
- Impuestos, financiación, apalancamiento real con margen y liquidaciones.
- Riesgo de contraparte del exchange, de custodia y de eventos extremos que no están en la muestra.
- Que la volatilidad o la correlación con el mercado cambien de régimen.
- La dependencia temporal y las colas pesadas, más allá de lo que conservan bloques de 5 días.

**¿Qué evidencia fuera de muestra sería necesaria?**
Un periodo nunca usado para elegir nada, con los parámetros congelados de antemano. Valoración diaria real de la cartera (no solo operaciones cerradas), con los costes de P08 aplicados una sola vez. Varios regímenes de mercado (subidas, caídas, laterales). Y suficientes operaciones independientes para que el Sharpe o el profit factor tengan intervalos de confianza útiles.

**No confundir métricas históricas con capacidad predictiva.** Un Sharpe o un profit factor pasados describen lo que ocurrió en la muestra; no garantizan que se repita.

📄 Resumen en `docs/conclusiones/P09_performance_risk.md`

</div>

Siguiente práctica: P10, valoración, tipos de interés y macroeconomía.